# Comparación de Telemetría — Robot Seguidor de Línea

Este notebook compara dos sesiones de datos del robot:

| | Archivo | Descripción |
|---|---|---|
| **CSV 1** | `telemetria_1781231215.csv` | Simulación con PID completo (P, I, D), posición x/y/theta, velocidad |
| **CSV 2** | `robot.csv` | Robot físico con PD (kp, kd), sensores s0–s3, PWM left/right |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta de colores por estado
COLOR_ESTADO = {
    'centrado':        '#2ecc71',
    'desviado':        '#e74c3c',
    'recuperacion':    '#f39c12',
    'vuelta_derecha':  '#3498db',
    'vuelta_izquierda':'#9b59b6',
}

print('Librerías cargadas ✓')

## 1. Carga de datos

In [ ]:
df1 = pd.read_csv('telemetria_1781231215.csv')
df2 = pd.read_csv('robot.csv')

# Normalizar nombre de columnas de PWM para comparar
# df1: pwm_l / pwm_r  →  df2: left / right
df1 = df1.rename(columns={'pwm_l': 'left', 'pwm_r': 'right'})

print('=== CSV 1 — Telemetría (simulación) ===')
print(f'  Filas: {len(df1):,}  |  Columnas: {list(df1.columns)}')
print(f'  Duración: {df1["t"].min():.3f}s → {df1["t"].max():.3f}s')
print()
print('=== CSV 2 — Robot físico ===')
print(f'  Filas: {len(df2):,}  |  Columnas: {list(df2.columns)}')
print(f'  Tests: {sorted(df2["test"].unique())}')

df1.head(3)

In [ ]:
df2.head(3)

## 2. Distribución de estados

In [ ]:
def resumen_estados(df, nombre):
    counts = df['estado'].value_counts()
    pcts   = df['estado'].value_counts(normalize=True) * 100
    resumen = pd.DataFrame({'conteo': counts, 'porcentaje': pcts.round(2)})
    resumen.index.name = 'estado'
    print(f'\n── {nombre} ──')
    print(resumen.to_string())
    return resumen

r1 = resumen_estados(df1, 'CSV 1 — Telemetría')
r2 = resumen_estados(df2, 'CSV 2 — Robot físico')

In [ ]:
# Estados comunes para comparar lado a lado
estados_comunes = ['centrado', 'desviado', 'recuperacion']
estados_solo_robot = ['vuelta_derecha', 'vuelta_izquierda']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, titulo in zip(axes, [df1, df2], ['CSV 1 — Telemetría (simulación)', 'CSV 2 — Robot físico']):
    counts = df['estado'].value_counts()
    colors = [COLOR_ESTADO.get(e, '#95a5a6') for e in counts.index]
    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=counts.index,
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.75,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax.set_title(titulo, fontweight='bold', pad=14)

plt.suptitle('Distribución de estados', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_estados_pie.png', bbox_inches='tight')
plt.show()

In [ ]:
# Comparación de estados comunes en barras agrupadas
pct1 = df1['estado'].value_counts(normalize=True).reindex(estados_comunes, fill_value=0) * 100
pct2 = df2['estado'].value_counts(normalize=True).reindex(estados_comunes, fill_value=0) * 100

x = np.arange(len(estados_comunes))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, pct1.values, w, label='CSV 1 — Telemetría', color='#2980b9', alpha=0.85)
bars2 = ax.bar(x + w/2, pct2.values, w, label='CSV 2 — Robot físico', color='#e67e22', alpha=0.85)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}%',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(estados_comunes, fontsize=11)
ax.set_ylabel('% de muestras')
ax.set_title('Comparación de estados comunes', fontweight='bold')
ax.legend()
ax.set_ylim(0, max(pct1.max(), pct2.max()) * 1.2)
plt.tight_layout()
plt.savefig('fig_estados_barras.png', bbox_inches='tight')
plt.show()

## 3. Análisis del error

In [ ]:
# Estadísticas de error
def stats_error(df, nombre):
    e = df['error']
    print(f'\n── Error — {nombre} ──')
    print(f'  Media:    {e.mean():.4f}')
    print(f'  Std:      {e.std():.4f}')
    print(f'  Min:      {e.min():.4f}')
    print(f'  Max:      {e.max():.4f}')
    print(f'  |error| medio: {e.abs().mean():.4f}')
    print(f'  % muestras |error| < 1: {(e.abs() < 1).mean()*100:.1f}%')
    print(f'  % muestras |error| < 5: {(e.abs() < 5).mean()*100:.1f}%')

stats_error(df1, 'CSV 1 — Telemetría')
stats_error(df2, 'CSV 2 — Robot físico')

In [ ]:
# Nota: las escalas de error son diferentes entre CSVs
# CSV1 usa error PID en escala amplia; CSV2 usa error normalizado pequeño
# Se grafican por separado con sus propias escalas

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

datasets = [
    (df1, 'CSV 1 — Telemetría', '#2980b9'),
    (df2, 'CSV 2 — Robot físico', '#e67e22'),
]

for col, (df, titulo, color) in enumerate(datasets):
    # Serie temporal del error
    ax = axes[0][col]
    x_axis = df['t'] if 't' in df.columns else np.arange(len(df))
    x_label = 'Tiempo (s)' if 't' in df.columns else 'Muestra'
    ax.plot(x_axis, df['error'], color=color, alpha=0.7, linewidth=0.8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.fill_between(x_axis, df['error'], 0, alpha=0.15, color=color)
    ax.set_title(f'{titulo}\nError a lo largo del tiempo', fontweight='bold')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Error')

    # Histograma del error
    ax2 = axes[1][col]
    ax2.hist(df['error'], bins=40, color=color, alpha=0.8, edgecolor='white')
    ax2.axvline(df['error'].mean(), color='black', linewidth=1.5,
                linestyle='--', label=f'Media: {df["error"].mean():.3f}')
    ax2.axvline(0, color='red', linewidth=1, linestyle=':', label='Error=0')
    ax2.set_title(f'{titulo}\nDistribución del error', fontweight='bold')
    ax2.set_xlabel('Error')
    ax2.set_ylabel('Frecuencia')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_error_temporal.png', bbox_inches='tight')
plt.show()

In [ ]:
# Error promedio por estado (en sus propias unidades)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (df, titulo, color) in zip(axes, datasets):
    grp = df.groupby('estado')['error'].agg(['mean', 'std']).reset_index()
    grp = grp.sort_values('mean')
    colors_bar = [COLOR_ESTADO.get(e, '#95a5a6') for e in grp['estado']]

    bars = ax.barh(grp['estado'], grp['mean'].abs(), color=colors_bar,
                   alpha=0.85, edgecolor='white')
    ax.errorbar(grp['mean'].abs(), grp['estado'],
                xerr=grp['std'], fmt='none', color='black', capsize=4, linewidth=1)

    for bar, val in zip(bars, grp['mean']):
        ax.text(bar.get_width() + grp['std'].max()*0.02,
                bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

    ax.set_title(f'{titulo}\n|Error| medio por estado (± std)', fontweight='bold')
    ax.set_xlabel('|Error| medio')

plt.tight_layout()
plt.savefig('fig_error_por_estado.png', bbox_inches='tight')
plt.show()

## 4. Comparación de señales de control (PWM)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, (df, titulo, color) in enumerate(datasets):
    x_axis = df['t'] if 't' in df.columns else np.arange(len(df))
    x_label = 'Tiempo (s)' if 't' in df.columns else 'Muestra'

    ax = axes[0][col]
    ax.plot(x_axis, df['left'],  label='Motor izquierdo', color='#8e44ad', alpha=0.8, linewidth=0.9)
    ax.plot(x_axis, df['right'], label='Motor derecho',   color='#16a085', alpha=0.8, linewidth=0.9)
    ax.set_title(f'{titulo}\nPWM motores en el tiempo', fontweight='bold')
    ax.set_xlabel(x_label)
    ax.set_ylabel('PWM')
    ax.legend(fontsize=8)

    ax2 = axes[1][col]
    diff = df['left'] - df['right']
    ax2.plot(x_axis, diff, color='#c0392b', alpha=0.7, linewidth=0.8)
    ax2.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax2.fill_between(x_axis, diff, 0, where=(diff > 0), alpha=0.2, color='#8e44ad', label='Gira izq.')
    ax2.fill_between(x_axis, diff, 0, where=(diff < 0), alpha=0.2, color='#16a085', label='Gira der.')
    ax2.set_title(f'{titulo}\nDiferencia PWM (izq − der)', fontweight='bold')
    ax2.set_xlabel(x_label)
    ax2.set_ylabel('ΔPWM')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_pwm.png', bbox_inches='tight')
plt.show()

In [ ]:
# PWM promedio por estado
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (df, titulo, _) in zip(axes, datasets):
    grp = df.groupby('estado')[['left', 'right']].mean().reset_index()
    x  = np.arange(len(grp))
    w  = 0.35

    ax.bar(x - w/2, grp['left'],  w, label='Motor izq.', color='#8e44ad', alpha=0.85)
    ax.bar(x + w/2, grp['right'], w, label='Motor der.', color='#16a085', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(grp['estado'], rotation=20, ha='right')
    ax.set_ylabel('PWM promedio')
    ax.set_title(f'{titulo}\nPWM medio por estado', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_pwm_estado.png', bbox_inches='tight')
plt.show()

## 5. Lecturas de sensores (s0–s3)

In [ ]:
sensores = ['s0', 's1', 's2', 's3']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, sensor in enumerate(sensores):
    ax = axes[i]

    # Boxplot por estado para cada CSV
    estados_1 = sorted(df1['estado'].unique())
    estados_2 = sorted(df2['estado'].unique())

    data1 = [df1[df1['estado'] == e][sensor].dropna().values for e in estados_1]
    data2 = [df2[df2['estado'] == e][sensor].dropna().values for e in estados_2]

    pos1 = np.arange(len(estados_1)) * 2
    pos2 = np.arange(len(estados_2)) * 2 + 1

    bp1 = ax.boxplot(data1, positions=pos1, widths=0.7,
                     patch_artist=True,
                     boxprops=dict(facecolor='#2980b9', alpha=0.6),
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='#2980b9'),
                     capprops=dict(color='#2980b9'),
                     flierprops=dict(marker='.', color='#2980b9', alpha=0.3, markersize=3))

    bp2 = ax.boxplot(data2, positions=pos2, widths=0.7,
                     patch_artist=True,
                     boxprops=dict(facecolor='#e67e22', alpha=0.6),
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='#e67e22'),
                     capprops=dict(color='#e67e22'),
                     flierprops=dict(marker='.', color='#e67e22', alpha=0.3, markersize=3))

    # Ticks en el centro de cada par
    n_max = max(len(estados_1), len(estados_2))
    all_estados = sorted(set(estados_1) | set(estados_2))
    tick_pos = [i * 2 + 0.5 for i in range(len(all_estados))]
    ax.set_xticks(tick_pos[:len(all_estados)])
    ax.set_xticklabels(all_estados, rotation=20, ha='right', fontsize=8)

    ax.set_title(f'Sensor {sensor.upper()}', fontweight='bold')
    ax.set_ylabel('Lectura')

    leg = [Patch(facecolor='#2980b9', alpha=0.7, label='CSV1 Telemetría'),
           Patch(facecolor='#e67e22', alpha=0.7, label='CSV2 Robot')]
    ax.legend(handles=leg, fontsize=7, loc='upper right')

plt.suptitle('Lecturas de sensores s0–s3 por estado', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_sensores.png', bbox_inches='tight')
plt.show()

## 6. Tabla resumen comparativa

In [ ]:
def tabla_resumen(df, nombre):
    rows = []
    for estado in df['estado'].unique():
        sub = df[df['estado'] == estado]
        row = {
            'CSV': nombre,
            'Estado': estado,
            'N muestras': len(sub),
            '% total': f"{len(sub)/len(df)*100:.1f}%",
            'Error medio': f"{sub['error'].mean():.4f}",
            'Error std': f"{sub['error'].std():.4f}",
            '|Error| medio': f"{sub['error'].abs().mean():.4f}",
            'PWM izq. medio': f"{sub['left'].mean():.2f}",
            'PWM der. medio': f"{sub['right'].mean():.2f}",
        }
        rows.append(row)
    return pd.DataFrame(rows)

t1 = tabla_resumen(df1, 'Telemetría')
t2 = tabla_resumen(df2, 'Robot físico')

tabla = pd.concat([t1, t2], ignore_index=True).sort_values(['Estado', 'CSV'])
tabla = tabla.reset_index(drop=True)

print('\n=== TABLA COMPARATIVA COMPLETA ===')
print(tabla.to_string(index=False))

In [ ]:
# Visualizar tabla como heatmap de % de tiempo por estado
pivot = pd.DataFrame({
    'Telemetría': df1['estado'].value_counts(normalize=True) * 100,
    'Robot físico': df2['estado'].value_counts(normalize=True) * 100,
}).fillna(0).round(2)

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=60)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=11)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = 'white' if val > 35 else 'black'
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                fontsize=12, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, label='% de muestras')
ax.set_title('Heatmap: % de tiempo por estado y CSV', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig_heatmap_estados.png', bbox_inches='tight')
plt.show()

print(pivot)

## 7. Transiciones de estado

In [ ]:
def matriz_transiciones(df, nombre):
    estados = df['estado'].values
    transiciones = pd.crosstab(
        pd.Series(estados[:-1], name='De'),
        pd.Series(estados[1:],  name='A'),
        normalize='index'
    ) * 100
    print(f'\n── Matriz de transiciones (%) — {nombre} ──')
    print(transiciones.round(1).to_string())
    return transiciones

m1 = matriz_transiciones(df1, 'CSV 1 — Telemetría')
m2 = matriz_transiciones(df2, 'CSV 2 — Robot físico')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (m, titulo) in zip(axes, [(m1, 'CSV 1 — Telemetría'), (m2, 'CSV 2 — Robot físico')]):
    im = ax.imshow(m.values, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks(range(len(m.columns)))
    ax.set_xticklabels(m.columns, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(m.index)))
    ax.set_yticklabels(m.index, fontsize=9)
    ax.set_xlabel('Estado siguiente')
    ax.set_ylabel('Estado actual')
    ax.set_title(f'{titulo}\nProbabilidad de transición (%)', fontweight='bold')

    for i in range(len(m.index)):
        for j in range(len(m.columns)):
            val = m.values[i, j]
            color = 'white' if val > 60 else 'black'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold', color=color)

    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('fig_transiciones.png', bbox_inches='tight')
plt.show()

## 8. Estabilidad: rachas de estados

In [ ]:
def analizar_rachas(df, nombre):
    estados = df['estado'].values
    rachas = []
    estado_actual = estados[0]
    longitud = 1

    for e in estados[1:]:
        if e == estado_actual:
            longitud += 1
        else:
            rachas.append({'estado': estado_actual, 'longitud': longitud})
            estado_actual = e
            longitud = 1
    rachas.append({'estado': estado_actual, 'longitud': longitud})

    df_rachas = pd.DataFrame(rachas)
    print(f'\n── Rachas por estado — {nombre} ──')
    resumen = df_rachas.groupby('estado')['longitud'].agg(['mean', 'median', 'max', 'count'])
    resumen.columns = ['Duración media', 'Mediana', 'Máx. racha', 'Num. rachas']
    print(resumen.round(2).to_string())
    return df_rachas

r1_rachas = analizar_rachas(df1, 'CSV 1 — Telemetría')
r2_rachas = analizar_rachas(df2, 'CSV 2 — Robot físico')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (df_r, titulo) in zip(axes, [
    (r1_rachas, 'CSV 1 — Telemetría'),
    (r2_rachas, 'CSV 2 — Robot físico')
]):
    grp = df_r.groupby('estado')['longitud'].mean().sort_values(ascending=True)
    colors_bar = [COLOR_ESTADO.get(e, '#95a5a6') for e in grp.index]
    bars = ax.barh(grp.index, grp.values, color=colors_bar, alpha=0.85, edgecolor='white')

    for bar, val in zip(bars, grp.values):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9)

    ax.set_title(f'{titulo}\nDuración media de racha (muestras)', fontweight='bold')
    ax.set_xlabel('Muestras consecutivas')

plt.tight_layout()
plt.savefig('fig_rachas.png', bbox_inches='tight')
plt.show()

## 9. Conclusiones

In [ ]:
print('=' * 60)
print('RESUMEN COMPARATIVO')
print('=' * 60)

for df, nombre in [(df1, 'CSV 1 — Telemetría'), (df2, 'CSV 2 — Robot físico')]:
    pct_centrado = (df['estado'] == 'centrado').mean() * 100
    pct_desviado = (df['estado'] == 'desviado').mean() * 100
    pct_recup    = (df['estado'] == 'recuperacion').mean() * 100
    err_abs      = df['error'].abs().mean()

    n_cambios = (df['estado'] != df['estado'].shift()).sum()
    tasa_cambio = n_cambios / len(df) * 100

    print(f'\n{nombre}')
    print(f'  Centrado:      {pct_centrado:.1f}%')
    print(f'  Desviado:      {pct_desviado:.1f}%')
    print(f'  Recuperación:  {pct_recup:.1f}%')
    print(f'  |Error| medio: {err_abs:.4f}')
    print(f'  Cambios de estado: {n_cambios} ({tasa_cambio:.1f}% de muestras)')

print()
print('Estado exclusivo del robot físico: vuelta_derecha, vuelta_izquierda')
pct_vd = (df2['estado'] == 'vuelta_derecha').mean() * 100
pct_vi = (df2['estado'] == 'vuelta_izquierda').mean() * 100
print(f'  vuelta_derecha:  {pct_vd:.1f}%')
print(f'  vuelta_izquierda: {pct_vi:.1f}%')
print('=' * 60)